<a href="https://colab.research.google.com/github/epi24/multimodal-meme-analysis/blob/main/image_only_clip_NEW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import CLIPModel, CLIPProcessor
from torchvision import transforms
from tqdm.auto import tqdm
from PIL import Image
import gc
from google.colab import drive



PATH_TRAIN_JSON = '/content/drive/MyDrive/meme_train.json'
PATH_VAL_JSON   = '/content/drive/MyDrive/meme_val.json'
PATH_IMAGES     = '/content/drive/MyDrive/all_memes/kym_memes'
SAVE_DIR        = '/content/drive/MyDrive/image_only_clip_NEW'
MODEL_NAME      = 'image_only_model_clip'



CHECKPOINT_PATH = None
START_EPOCH = 1
# HYPERPARAMETER
EPOCHS        = 20
LEARNING_RATE = 1e-5
BATCH_SIZE = 4500
NUM_WORKERS = 12
PREFETCH_FACTOR = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 2. DATASET (NUR BILDER)
# ==========================================
class ImageOnlyDataset(Dataset):
    def __init__(self, json_path, img_base_path, processor, is_train=True):
        self.processor = processor
        self.img_base_path = img_base_path
        self.samples = []
        self.label_map = {}
        self.id_to_label = {}

        # Augmentation nur fürs Training
        if is_train:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
            ])
        else:
            self.transform = None

        print(f"--- [DATASET] Lade {json_path.split('/')[-1]}... ---")

        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        # Labels ermitteln (alphabetisch sortieren für Konsistenz!)
        unique_labels = sorted(list(set(item['label'] for item in raw_data)))
        for idx, label in enumerate(unique_labels):
            self.label_map[label] = idx
            self.id_to_label[idx] = label

        # Map speichern (Nur einmal nötig, aber sicher ist sicher)
        with open(os.path.join(SAVE_DIR, f"{MODEL_NAME}_map.json"), 'w') as f:
            json.dump(self.id_to_label, f)

        for item in tqdm(raw_data, desc="Lade Bildpfade"):
            label_str = item.get('label')
            filename = item.get('filename')

            full_img_path = os.path.join(self.img_base_path, label_str, filename)

            # Nur Bilder nehmen, die auch physisch existieren
            if not os.path.exists(full_img_path):
                continue

            self.samples.append({
                'img_path': full_img_path,
                'label': self.label_map[label_str]
            })

        print(f"-> Bereit: {len(self.samples)} Bilder geladen.\n")
        del raw_data
        gc.collect()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        try:
            image = Image.open(sample['img_path']).convert("RGB")
            if self.transform:
                image = self.transform(image)
        except:
            # Fallback bei defektem Bild
            return self.__getitem__((idx + 1) % len(self.samples))

        # WICHTIG: Hier füttern wir dem Processor NUR das Bild.
        img_inputs = self.processor(images=image, return_tensors="pt")

        return {
            'pixel_values': img_inputs['pixel_values'].squeeze(0),
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

# ==========================================
# 3. ARCHITEKTUR (IMAGE-ONLY)
# ==========================================
class ImageOnlyNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

        # Backbone einfrieren, aber die letzte Vision-Schicht auftauen
        for param in self.clip.parameters():
            param.requires_grad = False
        for param in self.clip.vision_model.encoder.layers[-1].parameters():
            param.requires_grad = True
        for name, param in self.clip.named_parameters():
            if "layer_norm" in name:
                param.requires_grad = True

        # Classifier Head (Input ist 512, weil wir nur das Bild-Embedding haben)
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        # 1. Bild durch den Vision Encoder jagen
        vision_out = self.clip.vision_model(pixel_values=pixel_values)

        # 2. Die Projection Layer anwenden, um den finalen 512-Vektor zu kriegen
        img_embeds = self.clip.visual_projection(vision_out[1])

        # 3. Klassifizieren
        return self.classifier(img_embeds)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
def run_image_only_training():
    print("--- Start Image-Only Training ---")
    print(DEVICE)

    # Create the SAVE_DIR if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    train_dataset = ImageOnlyDataset(PATH_TRAIN_JSON, PATH_IMAGES, processor, is_train=True)
    val_dataset   = ImageOnlyDataset(PATH_VAL_JSON, PATH_IMAGES, processor, is_train=False)

    assert len(train_dataset.label_map) == len(val_dataset.label_map), "Train und Val haben unterschiedlich viele Klassen!"
    num_classes = len(train_dataset.label_map)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=PREFETCH_FACTOR)
    val_loader   = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=PREFETCH_FACTOR)

    model = ImageOnlyNet(num_classes).to(DEVICE)

    if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
        model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
        print(f"Checkpoint {CHECKPOINT_PATH} geladen.")
    else :
      print("beginne von neu")

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')

    print(f"Starte Training für {EPOCHS} Epochen...\n")

    for epoch in range(EPOCHS):
        # --- TRAINING ---
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoche {epoch+1} [Train]")

        for batch in pbar:
            optimizer.zero_grad()

            pixel_values = batch['pixel_values'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            with torch.amp.autocast('cuda'):
                logits = model(pixel_values) # Nur Pixel!
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        # Speicher aufräumen
        del pixel_values, labels, logits
        gc.collect()
        #torch.cuda.empty_cache()

        # --- VALIDIERUNG ---
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoche {epoch+1} [Valid]", leave=False):
                pixel_values = batch['pixel_values'].to(DEVICE)
                labels = batch['label'].to(DEVICE)

                with torch.amp.autocast('cuda'):
                    logits = model(pixel_values)

                _, preds = torch.max(logits, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

                del pixel_values, labels, logits

        val_acc = val_correct / val_total
        print(f" -> Resultat: Val Acc: {val_acc:.2%}")

        # --- SPEICHERN ---
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_NAME}_epoch_{START_EPOCH + epoch+1}.pth"))
        print("[SAVED] model")
        #if val_acc > best_acc:
         #   best_acc = val_acc
          #  torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_NAME}_{epoch+1}_best.pth"))
           # print(f"    [SAVED] Neues bestes Modell.")

if __name__ == "__main__":
    run_image_only_training()

--- Start Image-Only Training ---
cuda
--- [DATASET] Lade meme_train.json... ---


Lade Bildpfade:   0%|          | 0/75765 [00:00<?, ?it/s]

-> Bereit: 75763 Bilder geladen.

--- [DATASET] Lade meme_val.json... ---


Lade Bildpfade:   0%|          | 0/9402 [00:00<?, ?it/s]

-> Bereit: 9402 Bilder geladen.



...

Epoche 19 [Valid]:   0%|          | 0/3 [00:00<?, ?it/s]

 -> Resultat: Val Acc: 50.58%
[SAVED] model


Epoche 20 [Train]:   0%|          | 0/17 [00:00<?, ?it/s]

Epoche 20 [Valid]:   0%|          | 0/3 [00:00<?, ?it/s]

 -> Resultat: Val Acc: 51.55%
[SAVED] model


In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from tqdm.auto import tqdm
from PIL import Image
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
from google.colab import drive


CHECKPOINT_PATH = '/content/drive/MyDrive/image_only_clip_NEW/image_only_model_clip_epoch_20.pth' # Update with the actual epoch you want to evaluate
PATH_TEST_JSON   = '/content/drive/MyDrive/meme_test.json'
PATH_LABEL_MAP  = '/content/drive/MyDrive/image_only_clip_NEW/image_only_model_clip_map.json'
PATH_IMAGES     = '/content/drive/MyDrive/all_memes/kym_memes'
SAVE_DIR        = '/content/drive/MyDrive/image_only_clip_NEW'
OUTPUT_CSV      = 'clip_image_only_evaluation_results.csv'

BATCH_SIZE = 4500
NUM_WORKERS = 12
PREFETCH_FACTOR = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 2. DATASET (NUR BILDER + TEST-DATEN)
# ==========================================
class EvalClipImageDataset(Dataset):
    def __init__(self, json_path, img_base_path, label_map_path, processor):
        self.processor = processor
        self.img_base_path = img_base_path
        self.samples = []

        # --- LADE DIE OFFIZIELLE LABEL MAP ---
        print(f"Lade offizielle Label-Map: {label_map_path.split('/')[-1]}")
        with open(label_map_path, 'r', encoding='utf-8') as f:
            loaded_map = json.load(f)
            self.id_to_label = {int(k): v for k, v in loaded_map.items()}
            self.label_map = {v: int(k) for k, v in loaded_map.items()}

        print(f"--- [EVAL DATASET] Lade Test-Bilder aus {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        for item in tqdm(raw_data, desc="Lade Bildpfade"):
            label_str = item.get('label')
            filename = item.get('filename')

            # Unbekannte Klassen im Testset ignorieren
            if label_str not in self.label_map:
                continue

            full_img_path = os.path.join(self.img_base_path, label_str, filename)

            if not os.path.exists(full_img_path): continue

            self.samples.append({
                'img_path': full_img_path,
                'label': self.label_map[label_str],
                'filename': filename
            })

        print(f"-> Bereit: {len(self.samples)} Test-Bilder geladen.\n")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        try:
            image = Image.open(sample['img_path']).convert("RGB")
        except:
            # Fallback, falls ein Bild defekt ist
            return self.__getitem__((idx + 1) % len(self.samples))

        # CLIP Processor NUR für das Bild
        img_inputs = self.processor(images=image, return_tensors="pt")

        return {
            'pixel_values': img_inputs['pixel_values'].squeeze(0),
            'label': torch.tensor(sample['label'], dtype=torch.long),
            'filename': sample['filename']
        }

# ==========================================
# 3. ARCHITEKTUR (MUSS IDENTISCH ZUM TRAINING SEIN)
# ==========================================
class ClipImageOnlyNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

        # Classifier Head (Input ist 512, weil wir nur das Bild-Embedding haben)
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        vision_out = self.clip.vision_model(pixel_values=pixel_values)
        img_embeds = self.clip.visual_projection(vision_out[1])
        return self.classifier(img_embeds)

# ==========================================
# 4. EVALUATION LOOP
# ==========================================
def run_clip_image_evaluation():
    print("--- Starte CLIP Image-Only Test-Evaluation ---")

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    eval_dataset = EvalClipImageDataset(PATH_TEST_JSON, PATH_IMAGES, PATH_LABEL_MAP, processor)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, prefetch_factor=PREFETCH_FACTOR)

    num_classes = len(eval_dataset.label_map)
    model = ClipImageOnlyNet(num_classes).to(DEVICE)

    # --- GEWICHTE LADEN ---
    if os.path.exists(CHECKPOINT_PATH):
        print(f"Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            model.load_state_dict(checkpoint)
        print("[SUCCESS] Gewichte erfolgreich geladen.")
    else:
        print(f"[ERROR] Checkpoint nicht gefunden: {CHECKPOINT_PATH}")
        return

    model.eval()

    results_list = []
    all_preds = []
    all_labels = []

    print("Berechne Vorhersagen auf ungesehenen Daten...")

    with torch.no_grad():
        for batch in tqdm(eval_loader):
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            filenames = batch['filename']

            with torch.amp.autocast('cuda'):
                logits = model(pixel_values)

            probs = torch.softmax(logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i in range(len(filenames)):
                pred_idx = preds[i].item()
                true_idx = labels[i].item()

                results_list.append({
                    "Dateiname": filenames[i],
                    "Wahre Klasse": eval_dataset.id_to_label[true_idx],
                    "Vorhersage": eval_dataset.id_to_label[pred_idx],
                    "Status": "KORREKT" if pred_idx == true_idx else "FALSCH",
                    "Sicherheit_Prozent": round(confidences[i].item() * 100, 2)
                })

    # Gesamte Accuracy
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n========================================")
    print(f"CLIP IMAGE-ONLY ACCURACY (Test Set): {acc:.2%}")
    print(f"========================================\n")

    # Metriken berechnen
    class_names = [eval_dataset.id_to_label[i] for i in range(num_classes)]
    report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)

    # Detail-CSV speichern
    df_details = pd.DataFrame(results_list)
    os.makedirs(SAVE_DIR, exist_ok=True)
    save_path_csv = os.path.join(SAVE_DIR, OUTPUT_CSV)
    df_details.to_csv(save_path_csv, index=False, sep=';', encoding='utf-8-sig')

    # Metriken-CSV speichern
    metrics_list = []
    for name in class_names:
        metrics = report_dict[name]
        metrics_list.append({
            "Meme": name,
            "Precision": round(metrics['precision'], 2),
            "Recall": round(metrics['recall'], 2),
            "F1-Score": round(metrics['f1-score'], 2),
            "Anzahl": metrics['support']
        })

    df_metrics = pd.DataFrame(metrics_list)
    df_metrics = df_metrics.sort_values(by="F1-Score", ascending=True)
    df_metrics.to_csv(os.path.join(SAVE_DIR, "clip_image_only_metrics.csv"), index=False, sep=';')

    print("[FERTIG] Tabellen gespeichert.")

if __name__ == "__main__":
    run_clip_image_evaluation()


--- Starte CLIP Image-Only Test-Evaluation ---


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

Lade Checkpoint: /content/drive/MyDrive/image_only_clip_NEW/image_only_model_clip_epoch_20.pth
[SUCCESS] Gewichte erfolgreich geladen.
Berechne Vorhersagen auf ungesehenen Daten...


  0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in by


CLIP IMAGE-ONLY ACCURACY (Test Set): 51.05%

[FERTIG] Tabellen gespeichert.


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
